# 🌳 Preprocesamiento de Datos Espaciales  
## 🛰️ Construcción de la Base de Datos para Modelamiento de la Deforestación  

**Autor:** Andrea Natalia García Hernández  
**Proyecto:** Análisis de patrones puntuales para identificar los factores impulsores de la deforestación    
**Fecha:** Fecha de actualización: 14/02/2026      
**Institución:** Universidad Nacional de Colombia - Maestria en Geomática


## 🧭 1. Introducción

La preparación de datos constituye una fase fundamental del proceso de análisis espacial, ya que de ella depende la coherencia, calidad y reproducibilidad de los resultados obtenidos. En este trabajo, la preparación de los datos se orienta a la construcción de una base de datos espacial consistente y multitemporal, a partir de diferentes capas raster y vectoriales provenientes de diversas fuentes.

El proceso ha sido diseñado de manera **modular, reproducible y extensible**, utilizando herramientas de **programación SIG en Python**, lo que permite aplicar el mismo flujo metodológico a distintas capas temáticas (por ejemplo, cuerpos de agua, coberturas del suelo, restricciones territoriales), documentando cada una dentro de un marco común.

## 🎯 1. Objetivo

Describir el proceso de preparación, limpieza, homogeneización y estructuración de datos espaciales (vectoriales y raster) para la construcción de una base de datos geográfica destinada a la identificación de variables predictoras de la deforestación.


## 📦 2. Insumos

Los insumos utilizados se clasifican en cuatro categorías:
biofísicos, antrópicos, políticos-institucionales y económicos.

Todos los datasets fueron sometidos a procesos de:
- 📐 Estandarización de sistema de referencia espacial
- 🔄 Vectorización de capas raster
- ✂️ Recorte al área de estudio
- 🧩 Subdivisión espacial por unidades territoriales
- 🔄 Homogeneización de atributos
- 🧹 Limpieza topológica

### 🌎 2.1 Biofísicos

- 💧 Cuerpos de agua (MapBiomas Agua)
- 🗺️ Suelo (IGAC)
- 🌱 Capacidad de uso del suelo (IGAC)
- 🏞️ Coberturas y uso del suelo (MapBiomas Colombia, 2020–2024)

### 🏗️ 2.2 Antrópicos

- 🛣️ Vías terciarias (DNP)
- 🏠 Edificaciones (Google Open Buildings)
- 👥 Densidad poblacional (DANE)
- 📈 Proyección poblacional (DANE)
- ⛏️ Minería (MapBiomas)
- 🏙️ Infraestructura urbana
- 🌴 Palma aceitera
- 🌲 Silvicultura
- 🌾 Mosaicos agrícolas y pastos

### 🛡️ 2.3 Políticas de conservación

- 🌳 Áreas protegidas (RUNAP)
- 🪶 Resguardos indígenas (ANT)
- 🚜 Zonas de reserva campesina (ANT)
- 🌿 Bancos de hábitat (SIAC)

### 💰 2.4 Económicas

- 📊 Producto Interno Bruto (DANE)




## ⚙️ 3. Configuración del Entorno

In [1]:
# ===========================================
# 📚 Librerías principales
# ============================================

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from shapely.ops import unary_union
from pathlib import Path
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import importlib
import sys
import openpyxl
openpyxl.__version__

# ============================================
# 📁 Definición de rutas relativas
# ============================================

BASE = Path("..")
INPUT = BASE / "src"
sys.path.append(str(INPUT))

print("✅ Import correct")

✅ Import correct


## 🗺️ 4. Estandarización del Sistema de Referencia

Todos los datasets fueron reproyectados al sistema oficial
para garantizar coherencia en el cálculo de áreas y distancias.

In [7]:
# ============================================
# REPROYECTAR MULTIPLELS RASTER -> 9377
# ============================================

RASTER_DIR = BASE / "Datos" / "raster" / "agua_anual"  # Definir ruta de entrada
RASTER_OUT = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_anual_reproyectado"  # Definir ruta de salida
rasters = list(RASTER_DIR.glob("*.tif"))                                                                          

from data_preprocessing import reproyectar_raster                                                

for raster in rasters:
    salida = RASTER_OUT / raster.name.replace(".tif", "_9377.tif") # Definir nombre del raster de salida
    reproyectar_raster(raster, salida)

print(f"✅ Reproyectado: {salida.name}")

raster_reproj = list(RASTER_OUT.glob("*.tif"))

raster_path = list(RASTER_OUT.glob("*.tif"))[0]

with rasterio.open(raster_path) as src:                                                        
    print(src.count)      # número de bandas
    print(src.crs)        # sistema de coordenadas
    print(src.transform)  # transformación espacial

✅ Reproyectado: 2024_water_annual_water_bodies_18-1-1_6ffab515-c384-415b-b0fd-cb044e467d84_9377.tif
1
EPSG:9377
| 29.86, 0.00, 4023982.59|
| 0.00,-29.86, 3055621.14|
| 0.00, 0.00, 1.00|


In [2]:
# ============================================
# REPROYECTAR VECTOR -> 9377
# ============================================

from data_preprocessing import reproyectar_vector

VECTOR_IN = BASE / "Datos" / "vector" / "deforestacion" / "dashboard_alerts_shapefile.shp"  # Definir ruta de entrada
VECTOR_OUT = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_reproyectado" / "deforestacion_9377.gpkg" # Definir ruta de salida

reproyectar_vector(
    vector_entrada=VECTOR_IN,  
    vector_salida=VECTOR_OUT,
)

print("✅ Vector reproyectado correctamente")

✅ Vector reproyectado correctamente


## 🔄 5. Vectorización de capas raster

Las capas raster recortadas se convierten a formato vectorial con el fin de facilitar su integración en bases de datos espaciales, la realización de intersecciones espaciales y el cálculo preciso de métricas geométricas.

### 🛠️ 5.1 Procedimiento general

El proceso de vectorización consiste en:

🔸 Definir la condición o rango de valores del raster que representa el fenómeno de interés  
🔸 Identificar regiones contiguas de píxeles que cumplen dicha condición  
🔸 Generar geometrías poligonales a partir de estas regiones  

El resultado es una capa vectorial que representa espacialmente el fenómeno contenido en el raster para un año o periodo específico.

---


In [4]:
# ============================================
# VECTORIZAR MÚLTIPLES RASTER POR AÑO
# ============================================

RASTER_DIR = BASE / "agua_anual" / "resultados" / "agua_anual_reproyectado" # Definir ruta de entrada
raster_reproj = list(RASTER_DIR.glob("*.tif"))
VECTOR_OUT = BASE / "agua_anual" / "resultados" / "agua_vectorizada"

from data_preprocessing import raster_a_vector

for raster in rasters:
    anio = raster.name[:4]  # 2020, 2021, etc.
    
    salida = VECTOR_OUT / f"agua_{anio}.gpkg" # Definir nombre de ruta de salida
    
    raster_a_vector(
        raster_path=raster,
        vector_salida=salida,
        campo_valor="clase_agua",  # Definir nombre del campo donde se guardará el valor del píxel
        anio=anio,
        campo_anio="anio"
    )

print("✅ Raster vectorizado correctamente")

✅ Raster vectorizado correctamente


In [3]:
# ======================================================================================
# EXTRAER Y VECTORIZAR UN VALOR DE PIXEL EN LOS DIFERENTES AÑOS DE ESTUDIO
# ======================================================================================

from data_preprocessing import vectorizar_valor_pixel_por_anio

PIXEL_DIR = BASE / "Datos" / "Finales" / "coberturas"  # Definir ruta de entrada
rasters = sorted(PIXEL_DIR.glob("*.tif"))

gdfs = []

for raster in rasters:
    # extraer año del nombre
    year = int(raster.stem.split("-")[-1].split("_")[0])

    gdf_year = vectorizar_valor_pixel_por_anio(
        raster_path=raster,
        pixel_value=30,  # definir valor del pixel de interes
        year=year
    )

    gdfs.append(gdf_year)

gdf_pixel = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

gdf_pixel

out_dir = BASE / "Datos" / "vector" / "mineria" / "resultados" / "mineria_anual"  # Definir ruta de salida
out_dir.mkdir(parents=True, exist_ok=True)
salida = out_dir / "mineria_anual.gpkg" # Definir nombre de la capa de salida

gdf_pixel.to_file(
    salida,
    layer="mineria_anual", # Reescribir nombre de la capa de salida
    driver="GPKG"
)

print("✅ Vector creada con exito")

✅ Vector creada con exito


## ✂️ 6. Recorte espacial (clip)

### 🎯 6.1. Justificación

Dado que las capas raster y vectoriales suelen cubrir extensiones mayores al área de estudio, se realiza un recorte espacial previo para limitar el análisis exclusivamente a la zona de interés.

🔹 **Ventajas del recorte espacial**:
- Reducción del volumen de datos a procesar  
- Optimización del rendimiento computacional  
- Eliminación de información irrelevante

---

### ⚙️ 6.2. Implementación

El recorte espacial se realiza mediante funciones específicas, diferenciando entre capas raster y vectoriales:

- 🟦 **Capas raster**: el recorte se efectúa antes de la vectorización, utilizando el polígono del área de estudio como máscara.
- 🟩 **Capas vectoriales**: se aplica un recorte directo mediante operaciones de intersección espacial.

Este paso constituye una etapa común a todos los tipos de capas que se procesan dentro del flujo metodológico.


In [10]:
# ============================================
# CLIP DE MÚLTIPLES RASTER AL ÁREA DE ESTUDIO
# ============================================

from data_preprocessing import clip_a_area_estudio

VECTOR_DIR = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_vectorizada"  # Definir ruta de entrada
vectors = list(VECTOR_DIR.glob("*.gpkg"))

AREA_PATH = BASE / "Datos" / "vector" / "area_estudio" / "resultados" / "area_estudio_reproyectado" / "cuenca_guaviare_9377.gpkg"  # Definir ruta deL área de estudio
area_estudio = gpd.read_file(AREA_PATH)

CLIP_OUT = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_clip"   # Definir ruta de salida
CLIP_OUT.mkdir(parents=True, exist_ok=True)

for vector in vectors:
    print(f"✂️ Procesando {vector.name}")

    anio = vector.stem.split("_")[-1]

    gdf = gpd.read_file(vector)

    # Limpieza mínima
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[gdf.is_valid]

    salida = CLIP_OUT / f"agua_Guaviare_{anio}.gpkg" # Definir nombre de la capa de salida

    gdf_clip = clip_a_area_estudio(
        capa=gdf,
        area_estudio=area_estudio,
        guardar=True,
        ruta_salida=salida
    )

    print(f"✅ Guardado: {salida.name}")

✂️ Procesando agua_2020.gpkg
✅ Guardado: agua_Guaviare_2020.gpkg
✂️ Procesando agua_2021.gpkg
✅ Guardado: agua_Guaviare_2021.gpkg
✂️ Procesando agua_2022.gpkg
✅ Guardado: agua_Guaviare_2022.gpkg
✂️ Procesando agua_2023.gpkg
✅ Guardado: agua_Guaviare_2023.gpkg
✂️ Procesando agua_2024.gpkg
✅ Guardado: agua_Guaviare_2024.gpkg


In [3]:
# ============================================
# CLIP DE VECTOR AL ÁREA DE ESTUDIO
# ============================================

VECTOR_PATH = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_reproyectado" / "deforestacion_9377.gpkg" # Definir ruta de entrada
vector = gpd.read_file(VECTOR_PATH) 

AREA_PATH = BASE / "Datos" / "vector" / "area_estudio" / "resultados" / "area_estudio_reproyectado" / "cuenca_guaviare_9377.gpkg" # Definir ruta deL área de estudio
area_estudio = gpd.read_file(AREA_PATH)

from data_preprocessing import clip_a_area_estudio

CLIP_OUT = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_clip" # Definir ruta de salida
CLIP_OUT.mkdir(parents=True, exist_ok=True)

print("✂️ Recortando vector")

salida = CLIP_OUT / "deforestacion_clip.gpkg" # Definir nombre de la capa de salida

gdf_clip = clip_a_area_estudio(
    capa=vector,
    area_estudio=area_estudio,
    guardar=True,
    ruta_salida=salida
)

print(f"✅ Guardado: {salida.name}")

✂️ Recortando suelos
✅ Guardado: deforestacion_clip.gpkg


## 🧩 7. Subdivisión espacial por unidades territoriales

### 🎯 7.1. Justificación

Los fenómenos espaciales no respetan necesariamente los límites administrativos. Por ello, es necesario subdividir espacialmente las geometrías para:

✔️ Asignar correctamente los valores a cada unidad territorial  
✔️ Permitir análisis comparables a escalas administrativas  
✔️ Construir indicadores espaciales consistentes  

---

### ⚙️ 6.2. Implementación

La subdivisión espacial se realiza mediante operaciones de intersección entre:

- 🟩 las capas vectoriales temáticas generadas  
- 🟦 las capas de referencia territorial  

El resultado es una capa en la que cada geometría corresponde a la porción del fenómeno contenida dentro de una unidad territorial específica.

---

In [4]:
# ============================================
# SUBDIVIDIR UN VECTOR 
# ============================================

from data_preprocessing import subdividir_por_municipios

VECTOR_PATH = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_clip" / "deforestacion_clip.gpkg" # Definir ruta de entrada

OUT_VECTOR = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_muni" # Definir ruta de salida
OUT_VECTOR.mkdir(parents=True, exist_ok=True)

LIMITES_PATH = BASE / "Datos" / "Finales" / "municipio" / "municipio.gpkg" # Definir ruta de los límites territoriales con los que se va a subdividir la capa
limites = gpd.read_file(LIMITES_PATH)

print(f"🏘️ Subdividiendo vector")

gdf = gpd.read_file(VECTOR_PATH)

gdf_mun = subdividir_por_municipios(
    gdf=gdf,
    municipios=limites,
    campo_municipio="ID_MUNICIPIO" # Definir nombre del campo donde se guardará el ID
)

# 🔥 Separar multipolígonos en polígonos individuales
gdf = gdf.explode(index_parts=False).reset_index(drop=True)

salida = OUT_VECTOR / f"deforestacion_muni.gpkg" # Definir nombre de la capa de salida
gdf_mun.to_file(salida, driver="GPKG")
print(f"✅ Guardado: {salida.name}")

gdf_check = gpd.read_file(salida)
gdf_check.head()

🏘️ Subdividiendo vector


C:\Users\angar\anaconda3\envs\tesis_geomatica\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 95 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


✅ Guardado: deforestacion_muni.gpkg


,codealerta,fonte,biome,department,municipali,watershedl,regions,raisglimit,indigenous,forestres,natinatupr,areaha,anodetec,datadetec,dtimgant,dtimgdep,dtpubli,vpressao,ID_MUNICIPIO,geometry
0,40058686,{IDEAM},Amazonía,Vichada,Cumaribo,Cumaribo,Amazonia,Límite RAISG,(),(),(),1.12,2025.0,2025-03-31,2025-01-11,2025-03-26,2025-12-06,indigenous_traditional_cultural_use,99773,"MULTIPOLYGON Z (((5305980.826 1962084.923 0, 5..."
1,40054901,{IDEAM},Amazonía,Vichada,Cumaribo,Cumaribo,Amazonia,Límite RAISG,(),(),(),1.83,2024.0,2024-03-31,2023-12-20,2024-04-25,2025-12-10,farming,99773,"MULTIPOLYGON Z (((5284133.725 1967668.651 0, 5..."
2,40058701,{IDEAM},Amazonía,Vichada,Cumaribo,Cumaribo,Amazonia,Límite RAISG,(),(),(),1.79,2025.0,2025-03-31,2025-01-19,2025-03-26,2025-12-01,farming,99773,"MULTIPOLYGON Z (((5295736.451 1971640.576 0, 5..."
3,40058713,{IDEAM},Amazonía,Vichada,Cumaribo,Cumaribo,Amazonia,Límite RAISG,(),(),(),0.48,2025.0,2025-03-31,2024-12-02,2025-01-16,2025-12-01,indigenous_traditional_cultural_use,99773,"MULTIPOLYGON Z (((5286066.778 1972935.196 0, 5..."
4,40058718,{IDEAM},Amazonía,Vichada,Cumaribo,Cumaribo,Amazonia,Límite RAISG,(),(),(),0.44,2025.0,2025-03-31,2024-12-08,2025-01-12,2025-12-06,indigenous_traditional_cultural_use,99773,"MULTIPOLYGON Z (((5284893.709 1974724.11 0, 52..."


In [13]:
# ============================================
# SUBDIVIDIR MULTIPLES VECTOR
# ============================================

from data_preprocessing import subdividir_por_municipios

VECTOR_DIR = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_clip" # Definir ruta de entrada
vectores = list(VECTOR_OUT.glob("*.gpkg"))

OUT_DIR = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_municipio" # Definir ruta de salida
OUT_DIR.mkdir(parents=True, exist_ok=True)

LIMITES_PATH = BASE / "Datos" / "Finales" / "municipio" / "municipio.gpkg" # Definir ruta de los límites territoriales con los que se va a subdividir la capa
limites = gpd.read_file(LIMITES_PATH)

for vector in vectores:
    print(f"🏘️ Subdividiendo {vector.name}")

    anio = vector.stem.split("_")[-1]

    gdf = gpd.read_file(vector)

    gdf_mun = subdividir_por_municipios(
        gdf=gdf,
        municipios=limites,
        campo_municipio="ID_MUNICIPIO" # Definir nombre del campo donde se guardará el ID
    )

    salida = OUT_DIR / f"agua_municipio_{anio}.gpkg"  # Definir nombre de la capa de salida
    gdf_mun.to_file(salida, driver="GPKG")

    print(f"✅ Guardado: {salida.name}")


🏘️ Subdividiendo agua_mapb.gpkg
✅ Guardado: agua_municipio_mapb.gpkg


# 🌊 8. Unión y Consolidación de múltiples capas

## 🎯 Objetivo

Integrar las capas vectoriales anuales en una única capa histórica consolidada, garantizando la consistencia espacial y la unicidad de identificadores.

Esta capa permite analizar la dinámica temporal dentro del área de estudio y sirve como variable explicativa en el modelo predictivo de deforestación.

In [17]:
# ============================================
# UNIFICAR CAPAS VECTORIALES
# ============================================

from data_preprocessing import unir_capas_vectoriales

VECTOR_DIR = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_atrib" # Definir ruta de entrada

VECTOR_OUT = BASE / "Datos" / "Finales"/ "agua_anual" / "agua_anual.gpkg" # Definir ruta de salida
VECTOR_OUT.parent.mkdir(parents=True, exist_ok=True)

capas = list(VECTOR_DIR.glob("agua_atrib_*.gpkg")) # Definir nombre de las capas de la ruta de entrada

print(f"📂 Capas encontradas: {len(capas)}")

gdf = unir_capas_vectoriales(
    capas=capas,
    campo_id="ID_AGUA"  # Definir nombre del campo donde se guardará el ID
)

gdf.to_file(
    VECTOR_OUT,
    driver="GPKG"
)

print("✅ Capa histórica creada con ID único")

gdf["ANIO"].value_counts().sort_index() # Definir nombre del campo donde se guardó el año
gdf["ID_AGUA"].is_unique  # Definir nombre del campo donde se guardó el ID

📂 Capas encontradas: 5
✅ Capa histórica creada con ID único


True

## 🧹 9. Corrección Topológica

Se corrigieron posibles geometrías inválidas
para evitar inconsistencias en operaciones espaciales.


In [7]:
# ============================================
# ELIMINAR MULTIPOLIGONOS Y AGREGAR
# ============================================

VECTOR_DIR = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_muni" / "deforestacion_muni.gpkg"  # Definir ruta de entrada
gdf = gpd.read_file(VECTOR_DIR)

print("Antes:")
print(gdf.geom_type.value_counts())

# 2️⃣ Asegurar que no haya MultiPolygon previos
gdf = gdf.explode(index_parts=False).reset_index(drop=True)

# 3️⃣ Disolver por año (esto une los que se tocan)
# gdf_diss = gdf.dissolve(by="TIPO_RELIEVE")  # Definir nombre del campo donde se guardará el año

# 4️⃣ Volver a separar en Polygon individuales
gdf_final = gdf.explode(index_parts=False).reset_index()

print("\nDespués:")
print(gdf_final.geom_type.value_counts())

# 5️⃣ Guardar
VECTOR_OUT = BASE / "Datos"  / "vector" / "deforestacion" / "resultados" / "deforestacion_top" / "deforestacion_top.gpkg" # Definir ruta de salida
gdf_final.to_file(VECTOR_OUT)

print("Archivo guardado correctamente ✅")


Antes:
MultiPolygon    510
Name: count, dtype: int64

Después:
Polygon    1005
Name: count, dtype: int64
Archivo guardado correctamente ✅


In [103]:
# ============================================
# VERIFICAR GEOMETRÍAS INVALIDAS
# ============================================

gdt = BASE / "Datos" / "Finales" / "agua_anual" / "agua_anual.gpkg" 
vector = gpd.read_file(gdt)

# print(vector.crs)
vector.is_valid.value_counts()

True    241748
Name: count, dtype: int64

In [98]:
# ============================================
# IDENTIFICAR RAZON DE INVALIDADOS
# ============================================

from shapely.validation import explain_validity

vector["invalid_reason"] = vector.geometry.apply(explain_validity)

vector[vector.invalid_reason != "Valid Geometry"][["invalid_reason"]].head(12)

,invalid_reason
1738,Self-intersection[5270663.80338894 1863484.681...
1823,Self-intersection[5265106.60464324 1866969.570...
1827,Self-intersection[5259777.22812905 1873906.550...
1850,Self-intersection[5243575.72519581 1863161.215...
2117,Self-intersection[5302967.86417627 1904905.207...
3470,Self-intersection[5547489.5946779 1998027.3567...
3983,Self-intersection[5413410.02847183 1970605.213...
3985,Self-intersection[5412355.98184861 1969894.876...
3995,Self-intersection[5412893.77486612 1962263.141...
3996,Self-intersection[5410980.85603818 1964905.131...


In [99]:
# ============================================
# CORREGIR GEOMETRIAS INVALIDADAS
# ============================================

invalid_mask = ~vector.is_valid

vector.loc[invalid_mask, "geometry"] = (
    vector.loc[invalid_mask, "geometry"].buffer(0)
)

vector.is_valid.value_counts()
vector.geom_type.value_counts()

output_path = BASE / "Datos" / "Finales" / "Capacidad_uso" / "Capacidad_uso_1.gpkg" 
vector.to_file(output_path, driver="GPKG")

## 🧾 10. Definición y normalización de atributos

### 🧱 10.1. Estructura de la tabla de atributos

Con el fin de garantizar la homogeneidad entre las distintas capas preparadas, se define una estructura común de atributos que puede incluir:

- 🆔 Identificador único del registro  
- 🏘️ Identificador o nombre de la unidad territorial  
- 📅 Año o periodo de referencia  
- 📐 Métrica espacial de interés (por ejemplo, área)  
- 🏷️ Tipo de capa o fenómeno representado  

---

### 📏 10.2. Cálculo de métricas espaciales

Las métricas espaciales se calculan a partir de las geometrías vectoriales utilizando un sistema de referencia proyectado. Los resultados se expresan en unidades estándar (por ejemplo, kilómetros cuadrados) para facilitar la interpretación y el análisis comparativo.

---


In [5]:
# ============================================
# DEFINIR ATRIBUTOS PARA MULTIPLES VECTOR
# ============================================

INPUT_DIR = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_municipio"  # Definir ruta de entrada
vectores = list(INPUT_DIR.glob("*.gpkg"))
VECTOR_OUT = BASE / "Datos" / "raster" / "agua_anual" / "resultados" / "agua_atrib"  # Definir ruta de salida
VECTOR_OUT.mkdir(parents=True, exist_ok=True)

print(f"📂 Capas encontradas: {len(vectores)}")

from data_preprocessing import estandarizar_atributos

# ============================================
# DICCIONARIO DE CLASES
# ============================================

# Definir clases que corresponde al valor del pixel original (si aplica)
dicc_clase = {
    1: "Ríos, lagos, lagunas y ciénagas",
    2: "Reservorios",
    3: "Hidroeléctricas",
    4: "Minería",
    6: "Acuicultura"
}

# ============================================
# REGLAS DE ESTANDARIZACIÓN (MODELO DE DATOS)
# ============================================

# Definir nombre del campo final y nombre del campo original de donde llamará los datos
reglas = {
    "ID_AGUA": {"tipo": "id"},
    "TIPO_AGUA": {
        "desde": "clase_agua",
        "diccionario": dicc_clase
    },
    "ANIO": {"desde": "anio"},
    "ID_MUNICIPIO": {"desde": "ID_MUNICIPIO"}
}

# ============================================
# PROCESAMIENTO DE TODAS LAS CAPAS
# ============================================

for vector in vectores:
    print(f"Procesando {vector.name}")

    anio = vector.stem.split("_")[-1]

    gdf = gpd.read_file(vector)

    # -----------------------------
    # ESTANDARIZAR (PRIMERO)
    # -----------------------------
    gdf = estandarizar_atributos(
        gdf=gdf,
        reglas=reglas,
        iniciar_id=1
    )

    # -----------------------------
    # LIMPIEZA FINAL (DESPUÉS)
    # -----------------------------
    columnas_finales = [ # Definir nombre del campo final - Debe ser el mismo que se definió anteriormente
        "ID_AGUA",
        "TIPO_AGUA",
        "ANIO",
        "ID_MUNICIPIO",
        "geometry"
    ]

    gdf = gdf[columnas_finales]

    # -----------------------------
    # GUARDAR
    # -----------------------------
    salida = VECTOR_OUT / f"agua_atrib_{anio}.gpkg" # Definir nombre de las capas de salida
    gdf.to_file(salida, driver="GPKG")

    print(f"✅ Guardado: {salida.name}\n")

# ============================================
# VERIFICACIÓN
# ============================================

gdf_check = gpd.read_file(salida)
gdf_check.head()


📂 Capas encontradas: 6
Procesando agua_municipio_2020.gpkg
✅ Guardado: agua_atrib_2020.gpkg

Procesando agua_municipio_2021.gpkg
✅ Guardado: agua_atrib_2021.gpkg

Procesando agua_municipio_2022.gpkg
✅ Guardado: agua_atrib_2022.gpkg

Procesando agua_municipio_2023.gpkg
✅ Guardado: agua_atrib_2023.gpkg

Procesando agua_municipio_2024.gpkg
✅ Guardado: agua_atrib_2024.gpkg

Procesando agua_municipio_mapb.gpkg
✅ Guardado: agua_atrib_mapb.gpkg



,ID_AGUA,TIPO_AGUA,ANIO,ID_MUNICIPIO,geometry
0,1,None,mapb,99773,"MULTIPOLYGON Z (((-70.2683 4.16486 0, -70.2688..."
1,2,None,mapb,99773,"MULTIPOLYGON Z (((-70.26857 4.16404 0, -70.268..."
2,3,None,mapb,99773,"MULTIPOLYGON Z (((-70.26912 4.1635 0, -70.2691..."
3,4,Hidroeléctricas,mapb,99773,"MULTIPOLYGON Z (((-70.26858 4.16377 0, -70.268..."
4,5,Hidroeléctricas,mapb,99773,"MULTIPOLYGON Z (((-70.2694 4.15944 0, -70.2694..."


In [4]:
# ============================================
# DEFINIR ATRIBUTOS PARA UN VECTOR
# ============================================

VECTOR_DIR = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_top" / "deforestacion_top.gpkg" # Definir ruta de entrada
vector = gpd.read_file(VECTOR_DIR)
VECTOR_OUT = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_atr" # Definir ruta de salida
VECTOR_OUT.mkdir(parents=True, exist_ok=True)

print(f"📂 Capas encontradas: {len(vector)}")

if vector.crs.is_geographic:
    print("⚠️ Proyectando a Magna-Sirgas (EPSG:3116) para cálculo preciso de áreas...")
    vector = vector.to_crs(epsg=3116)

# Aseguramos que las columnas de fecha sean tipo datetime antes de procesar
vector['dtimgdep'] = pd.to_datetime(vector['dtimgdep'], errors='coerce')
vector['dtimgant'] = pd.to_datetime(vector['dtimgant'], errors='coerce')

from data_preprocessing import estandarizar_atributos

# ============================================
# REGLAS DE ESTANDARIZACIÓN (MODELO DE DATOS)
# ============================================

# Definir clases que corresponde al valor del pixel original (si aplica)
dicc_causa = {
    "agriculture": "Agricultura",
    "farming": "Agropecuario",
    "indigenous_traditional_cultural_use": "Uso cultural indígena",
    "livestock_farming": "Pecuario",
    "mining": "Minería",
    "others": "Otros",
    "road_opening": "Apertura de caminos",
    "windsweeper": "chagras de viento"
}

# Definir nombre del campo final y nombre del campo original de donde llamará los datos

reglas= {
    "ID_DEFO": {"tipo": "id"},
    "P_CAUSA":{"desde": "vpressao",
        "diccionario": dicc_causa
    },
    "ID_MUNICIPIO": {"desde": "ID_MUNICIPIO"},
    "ANIO":{"transformacion": lambda x: x['dtimgdep'].year},
    "MES":{"transformacion": lambda x: x['dtimgdep'].month},
    "ÁREA_HA":{"tipo": "area_ha"},
    "DURACION":{
        "transformacion": lambda x: (
            (x['dtimgdep'].year - x['dtimgant'].year) * 12 + 
            (x['dtimgdep'].month - x['dtimgant'].month)
        )
    }
}
    
# ============================================
# PROCESAMIENTO DE TODAS LAS CAPAS
# ============================================

print(f"Procesando vector")

gdf = estandarizar_atributos(
    gdf=vector,
    reglas=reglas,
    iniciar_id=1
)

columnas_finales = [ # Definir nombre del campo final - Debe ser el mismo que se definió anteriormente
    "ID_DEFO",
    "P_CAUSA",
    "ID_MUNICIPIO",
    "ANIO",
    "MES",
    "ÁREA_HA",
    "DURACION",
    "geometry"
]

gdf = gdf[columnas_finales]

# -----------------------------
# GUARDAR
# -----------------------------

salida = VECTOR_OUT/ f"deforestacion_atr.gpkg"
gdf.to_file(salida, driver="GPKG")
print(f"✅ Guardado: {salida.name}\n")

# ============================================
# VERIFICACIÓN RÁPIDA (OPCIONAL)
# ============================================

gdf_check = gpd.read_file(salida)
gdf_check.head()

📂 Capas encontradas: 1005
Procesando vector
✅ Guardado: deforestacion_atr.gpkg



,ID_DEFO,P_CAUSA,ID_MUNICIPIO,ANIO,MES,ÁREA_HA,DURACION,geometry
0,1,Uso cultural indígena,99773,2025,3,0.337943,2,"POLYGON Z ((5305980.826 1962084.923 0, 5305986..."
1,2,Uso cultural indígena,99773,2025,3,0.275295,2,"POLYGON Z ((5305870.038 1962075.64 0, 5305870...."
2,3,Uso cultural indígena,99773,2025,3,0.195535,2,"POLYGON Z ((5306290.07 1962164.375 0, 5306293...."
3,4,Uso cultural indígena,99773,2025,3,0.312498,2,"POLYGON Z ((5306133.194 1962208.61 0, 5306142...."
4,5,Agropecuario,99773,2024,4,1.829582,4,"POLYGON Z ((5284133.725 1967668.651 0, 5284139..."


## 📍 11. Conversión de Geometrías: Polígonos a Puntos Representativos

En este paso, transformamos las áreas de deforestación (polígonos) en puntos únicos. Esta simplificación es fundamental para análisis estadísticos espaciales, visualización de mapas de calor (heatmaps) o para alimentar modelos de aprendizaje automático donde se requiere una coordenada central por evento.

### ⚙️ Metodología: `Representative Point` vs `Centroid`

Para esta conversión, hemos optado por el método **Representative Point** en lugar del centroide tradicional por las siguientes razones técnicas:

1. **Garantía de Intersección:** A diferencia del centroide (centro de masas), el *punto representativo* garantiza que el punto resultante siempre caiga **dentro** de los límites del polígono original.
2. **Geometrías Complejas:** En parches de deforestación con formas de "C", "L" o anillos, el centroide matemático suele caer en el vacío (áreas de bosque no afectado). El punto representativo soluciona este error cartográfico.


In [3]:
# ============================================
# CONVERTIR POLÍGONOS DE DEFORESTACIÓN EN PUNTOS
# ============================================

VECTOR_DIR = BASE / "Datos" / "vector" / "deforestacion" / "resultados" / "deforestacion_atr" / "deforestacion_atr.gpkg" # Definir ruta de entrada
gdf_final = gpd.read_file(VECTOR_DIR)
VECTOR_OUT = BASE / "Datos" / "Finales" / "deforestacion"
VECTOR_OUT.mkdir(parents=True, exist_ok=True)

print(f"🔄 Convirtiendo {len(gdf_final)} polígonos a puntos...")

from data_preprocessing import poligonos_a_puntos

gdf_puntos = poligonos_a_puntos(gdf_final, metodo="representative")

ruta_salida_puntos = VECTOR_OUT / "deforestacion_puntos.gpkg"

# Guardar en formato GeoPackage
gdf_puntos.to_file(ruta_salida_puntos, driver="GPKG")

print(f"✅ Capa de puntos generada exitosamente.")
print(f"📍 Guardado en: {ruta_salida_puntos}")

print(f"Tipo de geometría resultante: {gdf_puntos.geometry.type.unique()}")
gdf_puntos.head()

🔄 Convirtiendo 1005 polígonos a puntos...
✅ Capa de puntos generada exitosamente.
📍 Guardado en: ..\Datos\Finales\deforestacion\deforestacion_puntos.gpkg
Tipo de geometría resultante: ['Point']


,ID_DEFO,P_CAUSA,ID_MUNICIPIO,ANIO,MES,ÁREA_HA,DURACION,geometry
0,1,Uso cultural indígena,99773,2025,3,0.337943,2,POINT (5306007.805 1962077.452)
1,2,Uso cultural indígena,99773,2025,3,0.275295,2,POINT (5305910.393 1962101.103)
2,3,Uso cultural indígena,99773,2025,3,0.195535,2,POINT (5306291.613 1962149.46)
3,4,Uso cultural indígena,99773,2025,3,0.312498,2,POINT (5306128.807 1962174.213)
4,5,Agropecuario,99773,2024,4,1.829582,4,POINT (5284191.873 1967634.589)
